# Kai RNN Recons Training (Self-Contained)

This notebook is a self-contained training pipeline for the recons RNN model.
It inlines the key logic from `Playground_Kai/train_recons.py`, `Playground_Kai/data_utils.py`, and the relevant `emg2qwerty` helpers (charset/labels/decoder/CER).

Run all cells top-to-bottom.

Outputs:
- Best checkpoint: `Playground_Kai/checkpoints/best_rnn_recons_raw_notebook.pt`
- Final test metrics + a few true/pred examples

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'pyyaml', 'h5py', 'matplotlib'], check=True)

import json
import math
import string
import time
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
import yaml
from torch import nn
from torch.utils.data import ConcatDataset, DataLoader, Dataset

torch.set_float32_matmul_precision('high')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

In [ ]:
# Best hyperparameters copied from Playground_Kai/checkpoints/best_hyperparams_rnn_recons_raw_backup.yaml
BEST_HP = {
    'lr': 0.0008215117598353862,
    'weight_decay': 0.00935611311371853,
    'proj_dim': 64,
    'hidden_size': 256,
    'num_layers': 1,
    'dropout': 0.11750003567529757,
}

# Training control
EPOCHS = 150
BATCH_SIZE = 32
WINDOW_LENGTH = 500
PADDING = (10, 2)
NUM_WORKERS = 0
MIN_LR = 1e-5
WARMUP_EPOCHS = 5
SHOW_EXAMPLES = 5

# Resolve workspace root robustly whether notebook runs from root or Playground_Kai/
CWD = Path.cwd().resolve()
ROOT = CWD if (CWD / 'Playground_Kai').exists() else CWD.parent
DATA_ROOT = ROOT / 'data_recons'
CONFIG_PATH = ROOT / 'config' / 'user' / 'single_user.yaml'
CKPT_DIR = ROOT / 'Playground_Kai' / 'checkpoints'
CKPT_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = CKPT_DIR / 'best_rnn_recons_raw_notebook.pt'

print('ROOT:', ROOT)
print('DATA_ROOT exists:', DATA_ROOT.exists())
print('CONFIG_PATH exists:', CONFIG_PATH.exists())
print('Checkpoint path:', CKPT_PATH)

In [ ]:
# ---- Minimal charset + label helpers (self-contained replacement for emg2qwerty.charset / LabelData) ----
KEY_TO_CHAR = {
    **{c: c for c in (string.ascii_letters + string.digits + string.punctuation)},
    'Key.backspace': '\b',
    'Key.enter': '\n',
    'Key.space': ' ',
    'Key.shift': '^',
}
KEY_ORDER = list(string.ascii_letters + string.digits + string.punctuation) + [
    'Key.backspace', 'Key.enter', 'Key.space', 'Key.shift'
]
KEY_TO_LABEL = {k: i for i, k in enumerate(KEY_ORDER)}
LABEL_TO_KEY = {i: k for k, i in KEY_TO_LABEL.items()}
NULL_CLASS = len(KEY_ORDER)
NUM_CLASSES = len(KEY_ORDER) + 1

def normalize_key(raw_key: str) -> str | None:
    if raw_key in KEY_TO_LABEL:
        return raw_key
    aliases = {
        'Key.shift_l': 'Key.shift',
        'Key.shift_r': 'Key.shift',
        '\n': 'Key.enter',
        '\r': 'Key.enter',
        ' ': 'Key.space',
    }
    if raw_key in aliases and aliases[raw_key] in KEY_TO_LABEL:
        return aliases[raw_key]
    return raw_key if raw_key in KEY_TO_LABEL else None

def labels_to_text(labels: np.ndarray | list[int]) -> str:
    out = []
    for x in labels:
        xi = int(x)
        if xi == NULL_CLASS:
            continue
        key = LABEL_TO_KEY.get(xi)
        if key is None:
            continue
        out.append(KEY_TO_CHAR.get(key, ''))
    return ''.join(out)

def labels_from_keystrokes(keystrokes: list[dict], start_t: float, end_t: float) -> torch.Tensor:
    labels: list[int] = []
    for k in keystrokes:
        t = float(k.get('start', -1e18))
        if t > end_t:
            break
        if t >= start_t:
            nk = normalize_key(str(k.get('key', '')))
            if nk is not None:
                labels.append(KEY_TO_LABEL[nk])
    return torch.tensor(labels, dtype=torch.long)

def greedy_decode_labels(emissions_tnc: torch.Tensor, emission_lengths: torch.Tensor) -> list[str]:
    emissions = emissions_tnc.detach().cpu()
    T, N, _ = emissions.shape
    pred_ids = emissions.argmax(dim=-1).numpy()
    out_text: list[str] = []
    for n in range(N):
        L = int(emission_lengths[n])
        seq = pred_ids[:L, n].tolist()
        collapsed = []
        prev = None
        for x in seq:
            if x != prev and x != NULL_CLASS:
                collapsed.append(x)
            prev = x
        out_text.append(labels_to_text(collapsed))
    return out_text

def edit_distance(a: str, b: str) -> int:
    R, H = len(a), len(b)
    dp = np.zeros((R + 1, H + 1), dtype=np.int32)
    dp[:, 0] = np.arange(R + 1)
    dp[0, :] = np.arange(H + 1)
    for r in range(1, R + 1):
        for h in range(1, H + 1):
            if a[r - 1] == b[h - 1]:
                dp[r, h] = dp[r - 1, h - 1]
            else:
                dp[r, h] = 1 + min(dp[r - 1, h], dp[r, h - 1], dp[r - 1, h - 1])
    return int(dp[R, H])

def cer_percent(references: list[str], predictions: list[str]) -> float:
    total_edits = 0
    total_chars = 0
    for ref, hyp in zip(references, predictions):
        total_edits += edit_distance(ref, hyp)
        total_chars += max(len(ref), 1)
    return 100.0 * total_edits / max(total_chars, 1)

print('num classes:', NUM_CLASSES, '| blank id:', NULL_CLASS)

In [ ]:
# ---- Recons dataset + loaders (self-contained replacement for Playground_Kai.data_utils recons section) ----
def get_recons_session_paths(data_root: Path, config_path: Path) -> dict[str, list[Path]]:
    with open(config_path, 'r', encoding='utf-8') as f:
        cfg = yaml.safe_load(f)
    out: dict[str, list[Path]] = {}
    for split in ('train', 'val', 'test'):
        paths = []
        for entry in cfg['dataset'].get(split, []):
            session = entry['session']
            p = data_root / f'{session}_recons_v3.hdf5'
            if not p.exists():
                raise FileNotFoundError(f'Missing recons file: {p}')
            paths.append(p)
        out[split] = paths
    return out

class ReconsRawDataset(Dataset):
    def __init__(self, hdf5_path: Path, window_length: int = 500, stride: int | None = None, padding: tuple[int, int] = (10, 2), jitter: bool = False):
        self.hdf5_path = hdf5_path
        self.window_length = int(window_length)
        self.stride = int(window_length if stride is None else stride)
        self.left_padding, self.right_padding = int(padding[0]), int(padding[1])
        self.jitter = bool(jitter)
        self._file = None

        with h5py.File(hdf5_path, 'r') as f:
            grp = f['emg2qwerty']
            self.session_length = int(grp['timeseries/emg_left'].shape[0])
            self.keystrokes = json.loads(grp.attrs.get('keystrokes', '[]'))

        if self.window_length <= 0 or self.stride <= 0:
            raise ValueError('window_length and stride must be > 0')

    def _ensure_open(self):
        if self._file is None:
            self._file = h5py.File(self.hdf5_path, 'r')

    def __len__(self) -> int:
        return int(max(self.session_length - self.window_length, 0) // self.stride + 1)

    def __getitem__(self, idx: int):
        self._ensure_open()
        ts = self._file['emg2qwerty/timeseries']
        offset = int(idx * self.stride)

        leftover = self.session_length - (offset + self.window_length)
        if leftover < 0:
            raise IndexError(idx)
        if leftover > 0 and self.jitter:
            offset += int(np.random.randint(0, min(self.stride, leftover)))

        window_start = max(offset - self.left_padding, 0)
        window_end = min(offset + self.window_length + self.right_padding, self.session_length)

        emg_left = ts['emg_left'][window_start:window_end]
        emg_right = ts['emg_right'][window_start:window_end]
        times = ts['time'][window_start:window_end]

        emg_tensor = torch.stack([
            torch.from_numpy(np.asarray(emg_left, dtype=np.float32)),
            torch.from_numpy(np.asarray(emg_right, dtype=np.float32)),
        ], dim=1)

        start_t = float(times[offset - window_start])
        end_t = float(times[(offset + self.window_length - 1) - window_start])
        labels = labels_from_keystrokes(self.keystrokes, start_t, end_t)
        return emg_tensor, labels

    @staticmethod
    def collate(samples):
        inputs = [s[0] for s in samples]
        targets = [s[1] for s in samples]
        input_batch = nn.utils.rnn.pad_sequence(inputs)
        target_batch = nn.utils.rnn.pad_sequence(targets)
        input_lengths = torch.tensor([len(x) for x in inputs], dtype=torch.int32)
        target_lengths = torch.tensor([len(t) for t in targets], dtype=torch.int32)
        return {
            'inputs': input_batch,
            'targets': target_batch,
            'input_lengths': input_lengths,
            'target_lengths': target_lengths,
        }

def get_recons_dataloaders(data_root: Path, config_path: Path, window_length: int = 500, stride: int | None = None, padding: tuple[int, int] = (10, 2), batch_size: int = 32, num_workers: int = 0):
    sp = get_recons_session_paths(data_root, config_path)
    train_dataset = ConcatDataset([
        ReconsRawDataset(p, window_length=window_length, stride=stride, padding=padding, jitter=True)
        for p in sp['train']
    ])
    val_dataset = ConcatDataset([
        ReconsRawDataset(p, window_length=window_length, stride=None, padding=padding, jitter=False)
        for p in sp['val']
    ])
    test_dataset = ConcatDataset([
        ReconsRawDataset(p, window_length=window_length, stride=None, padding=padding, jitter=False)
        for p in sp['test']
    ])

    persistent = num_workers > 0
    loaders = {
        'train': DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, collate_fn=ReconsRawDataset.collate, pin_memory=True, persistent_workers=persistent),
        'val': DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, collate_fn=ReconsRawDataset.collate, pin_memory=True, persistent_workers=persistent),
        'test': DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=num_workers, collate_fn=ReconsRawDataset.collate, pin_memory=True, persistent_workers=persistent),
    }

    print(f"Recons raw dataset | train: {len(sp['train'])} sessions ({len(train_dataset)} windows) | val: {len(sp['val'])} sessions ({len(val_dataset)} windows) | test: {len(sp['test'])} sessions ({len(test_dataset)} windows)")
    return loaders

loaders = get_recons_dataloaders(
    data_root=DATA_ROOT,
    config_path=CONFIG_PATH,
    window_length=WINDOW_LENGTH,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    padding=PADDING,
)
print('train batches:', len(loaders['train']), '| val batches:', len(loaders['val']), '| test batches:', len(loaders['test']))

In [ ]:
# ---- Recons RNN model (self-contained replacement for Playground_Kai.model ReconsRNNEncoder) ----
class ReconsRNNEncoder(nn.Module):
    NUM_BANDS = 2

    def __init__(self, in_channels: int = 16, proj_dim: int = 64, hidden_size: int = 256, num_layers: int = 1, dropout: float = 0.11750003567529757):
        super().__init__()
        self.input_norm = nn.LayerNorm(in_channels)
        self.proj = nn.Linear(in_channels, proj_dim)
        self.proj_norm = nn.LayerNorm(proj_dim)
        rnn_in = self.NUM_BANDS * proj_dim
        self.rnn = nn.LSTM(
            input_size=rnn_in,
            hidden_size=hidden_size,
            num_layers=num_layers,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
            batch_first=False,
        )
        self.head = nn.Linear(2 * hidden_size, NUM_CLASSES)

    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        # inputs: (T, N, 2, 16)
        x = self.input_norm(inputs)
        x = self.proj(x)
        x = self.proj_norm(x)
        x = x.flatten(start_dim=2)
        x = x.contiguous()
        x, _ = self.rnn(x)
        x = self.head(x)
        return F.log_softmax(x, dim=-1)

model = ReconsRNNEncoder(
    in_channels=16,
    proj_dim=BEST_HP['proj_dim'],
    hidden_size=BEST_HP['hidden_size'],
    num_layers=BEST_HP['num_layers'],
    dropout=BEST_HP['dropout'],
).to(device)
print('model params:', sum(p.numel() for p in model.parameters()))

In [ ]:
# ---- Train / eval loop (self-contained replacement for train_recons main flow) ----
def lr_lambda(step: int, warmup_steps: int, total_steps: int, min_lr_ratio: float) -> float:
    if step < warmup_steps:
        return float(step) / max(1, warmup_steps)
    progress = float(step - warmup_steps) / max(1, total_steps - warmup_steps)
    cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
    return min_lr_ratio + (1.0 - min_lr_ratio) * cosine

def train_one_epoch(model, loader, optimizer, criterion, scheduler):
    model.train()
    total_loss = 0.0
    for batch in loader:
        inputs = batch['inputs'].to(device)
        targets = batch['targets'].to(device)
        input_lengths = batch['input_lengths'].to(device)
        target_lengths = batch['target_lengths'].to(device)

        optimizer.zero_grad()
        emissions = model(inputs)
        loss = criterion(emissions, targets.transpose(0, 1), input_lengths, target_lengths)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()
        scheduler.step()

        total_loss += float(loss.item())
    return total_loss / max(len(loader), 1)

@torch.no_grad()
def evaluate(model, loader, criterion, preview_limit: int = 0):
    model.eval()
    total_loss = 0.0
    refs: list[str] = []
    hyps: list[str] = []
    previews: list[tuple[str, str]] = []

    for batch in loader:
        inputs = batch['inputs'].to(device)
        targets = batch['targets']
        input_lengths = batch['input_lengths']
        target_lengths = batch['target_lengths']

        emissions = model(inputs)
        loss = criterion(emissions, targets.transpose(0, 1).to(device), input_lengths.to(device), target_lengths.to(device))
        total_loss += float(loss.item())

        pred_texts = greedy_decode_labels(emissions, input_lengths)
        targets_np = targets.numpy()
        tgt_lens = target_lengths.numpy()

        for i in range(len(pred_texts)):
            ref_text = labels_to_text(targets_np[: int(tgt_lens[i]), i])
            hyp_text = pred_texts[i]
            refs.append(ref_text)
            hyps.append(hyp_text)
            if preview_limit > 0 and len(previews) < preview_limit:
                previews.append((ref_text, hyp_text))

    cer = cer_percent(refs, hyps)
    metrics = {'CER': cer}
    return total_loss / max(len(loader), 1), metrics, previews

criterion = nn.CTCLoss(blank=NULL_CLASS, zero_infinity=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=BEST_HP['lr'], weight_decay=BEST_HP['weight_decay'])
steps_per_epoch = len(loaders['train'])
total_steps = EPOCHS * steps_per_epoch
warmup_steps = WARMUP_EPOCHS * steps_per_epoch
min_lr_ratio = MIN_LR / BEST_HP['lr']
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lambda s: lr_lambda(s, warmup_steps, total_steps, min_lr_ratio))

best_cer = float('inf')
train_losses = []
val_losses = []
val_cers = []

print(f'\nTraining RNN for {EPOCHS} epochs...')
for epoch in range(EPOCHS):
    t0 = time.perf_counter()
    tr_loss = train_one_epoch(model, loaders['train'], optimizer, criterion, scheduler)
    va_loss, va_metrics, _ = evaluate(model, loaders['val'], criterion, preview_limit=0)
    va_cer = va_metrics['CER']

    train_losses.append(tr_loss)
    val_losses.append(va_loss)
    val_cers.append(va_cer)

    if va_cer < best_cer:
        best_cer = va_cer
        torch.save({
            'epoch': epoch,
            'model': model.state_dict(),
            'optimizer': optimizer.state_dict(),
            'scheduler': scheduler.state_dict(),
            'best_cer': best_cer,
            'hp': BEST_HP,
        }, CKPT_PATH)
        saved = '  [saved best]'
    else:
        saved = ''

    dt = time.perf_counter() - t0
    print(f"Epoch {epoch + 1:3d}/{EPOCHS} | train_loss={tr_loss:.4f} | val_loss={va_loss:.4f} | val_CER={va_cer:.2f}% | {dt:.1f}s{saved}")

print('Best val CER:', best_cer)
print('Best checkpoint:', CKPT_PATH)

In [ ]:
# ---- Final test evaluation + sample predictions + curves ----
ckpt = torch.load(CKPT_PATH, map_location=device)
model.load_state_dict(ckpt['model'])
print('Loaded best checkpoint from epoch', int(ckpt['epoch']) + 1)

test_loss, test_metrics, previews = evaluate(model, loaders['test'], criterion, preview_limit=SHOW_EXAMPLES)
print(f"Test loss: {test_loss:.4f}")
print(f"Test CER : {test_metrics['CER']:.2f}%")

if previews:
    print(f'\nSample test predictions (first {len(previews)}):')
    for i, (ref, hyp) in enumerate(previews, 1):
        print(f'  [{i}] true: {ref}')
        print(f'      pred: {hyp}')

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(train_losses, label='train')
ax[0].plot(val_losses, label='val')
ax[0].set_title('CTC Loss')
ax[0].set_xlabel('Epoch')
ax[0].grid(True, lw=0.3)
ax[0].legend()

ax[1].plot(val_cers, label='val CER')
ax[1].set_title('Validation CER (%)')
ax[1].set_xlabel('Epoch')
ax[1].grid(True, lw=0.3)
ax[1].legend()

plt.tight_layout()
plt.show()